In [1]:
import pandas as pd
import os 
from pathlib import Path


# make sample yaml for GC correction 

In [2]:
# Extract the path of a matched normal for a given sample - use POGID if avaialble, or PAN ID if not 
def get_WBC_Path(sample_id):
    df = pd.read_csv('/projects/pangen/analysis/jtopham/wgs_paths.tsv', sep='\t',
                     names=['pangen_id', 'biopsy', 'other', 'path'])
    df = df[df['biopsy'] == 'normal']
    df = df[['pangen_id', 'path']]
    
    if sample_id not in df['pangen_id'].values:
        raise ValueError(f"No WBC path found for sample {sample_id}")
    
    return df[df['pangen_id'] == sample_id]['path'].unique()[0]

# Normalize path for WBC and check if its a cram instead of a bam/viceversa
def resolve_wgs_path(p: str):
    # remove double slashes and normalize path 
    p = os.path.normpath(p)
    base, ext = os.path.splitext(p)
    for ext in ['.bam', '.cram']:
        cand = base + ext
        if os.path.exists(cand):
            return cand


In [4]:
# Read in a dataframe for all our POG IDs with 
samples = pd.read_csv('/projects/pangen/analysis/hmac/data/ctf_correlations/data/ichor_metrics_with_PAN_POG_IDs.csv')
samples = samples['sample'].tolist()
samples = samples + ['POG998', 'POG1008']

In [ ]:
# Make YAML file with sample for Griffin GC bias correction 

outfile = Path('/projects/pangen/analysis/hmac/applications/Griffin/haley_scripts/testing_scripts/samples.yaml')

with outfile.open("w") as file:
    file.write("samples:\n")
    for sample in samples: 
        tumour_path = f'/projects/pangen/analysis/jtopham/samples/{sample}/frag/bams/{sample}-cfDNA.bam'
        file.write(f"    {sample}: {tumour_path}\n")
        


K now run Griffin GC correction snakemake 

# make sample yaml for nucleosome profiling 

In [5]:
# Make YAML file with sample for Griffin GC bias correction 

outfile = Path('/projects/pangen/analysis/hmac/applications/Griffin/snakemakes/griffin_nucleosome_profiling/config/samples.GC.yaml')

with outfile.open("w") as file:
    file.write("samples:\n")
    for sample in samples: 
        file.write(f"   {sample}:\n")
        tumour_path = f'/projects/pangen/analysis/jtopham/samples/{sample}/frag/bams/{sample}-cfDNA.bam'
        file.write(f"       bam: {tumour_path}\n")
        file.write(f"       GC_bias: /projects/pangen/analysis/hmac/applications/Griffin/snakemakes/griffin_GC_and_mappability_correction/results/GC_bias/{sample}.GC_bias.txt\n")